In [ ]:
%%writefile eikonale.h
#pragma once           // il file viene incluso una sola volta
// Risolutore locale 3D (Godunov) ed errore rispetto alla soluzione esatta della sfera
#include <vector>
#include <cmath>      // fminf, sqrtf
const float INF = 1e30f;                 // "infinito": nodo non ancora raggiunto dall'onda

// Risolutore locale 3D (Godunov, Alg. 2 esteso al 3D): dai minimi dei vicini a, b, c al nuovo u
// Prova con 1, 2, poi 3 vicini; usato dal Block FIM (GPU)
__host__ __device__ float solve_quadratic3(float a, float b, float c,float F, float h)
{
    float Fh = F * h;                              // tempo per attraversare una cella
    if (a > b) { float t = a; a = b; b = t; }      // ordina: a <= b <= c
    if (b > c) { float t = b; b = c; c = t; }
    if (a > b) { float t = a; a = b; b = t; }

    float u = a + Fh;                              // 1 vicino (caso 1)
    if (u <= b) return u;
    u = 0.5f * (a + b + sqrtf(2.0f * Fh * Fh - (a - b) * (a - b)));   // 2 vicini (caso 2 = Alg. 2)
    if (u <= c) return u;
    float d = 3.0f * Fh * Fh - ((a - b) * (a - b) + (b - c) * (b - c) + (c - a) * (c - a));
    return (a + b + c + sqrtf(d)) / 3.0f;          // 3 vicini (caso 3)                                  
}

// Errore relativo ||u - U|| / ||U|| rispetto alla soluzione esatta U = distanza dal centro
double relative_error(const std::vector<float>& u, int N, float h)
{
    double num = 0.0, den = 0.0;
    for (int k = 0; k < N; ++k)
        for (int i = 0; i < N; ++i)
            for (int j = 0; j < N; ++j) {
            double x = -1.0 + j * h, y = -1.0 + i * h, z = -1.0 + k * h;   // coordinate del nodo (k, i, j)
            double U = std::sqrt(x * x + y * y + z * z);        // soluzione esatta
            double d = u[(k * N + i) * N + j] - U;
            num += d * d;                               // ||u - U||^2
            den += U * U;                               // ||U||^2
            }
    return std::sqrt(num / den);    // Norma euclidea relativa: ||u - U|| / ||U||
}

In [ ]:
%%writefile block_fim.cuh
#pragma once           // il file viene incluso una sola volta
// Block FIM (Alg. 7) sulla GPU: blocchi 8^3 in shared memory, sorgente singola, nessun ostacolo
#include <cstdio>
#include <cstdlib>
#include <vector>
#include <algorithm>       // std::min
#include <cuda_runtime.h>
#include "eikonale.h"         // INF e solve_quadratic3

// Controllo degli errori CUDA: se una chiamata fallisce, stampa dove e perche' ed esce
#define CUDA_CHECK(call)                                                     \
    do {                                                                     \
        cudaError_t err = (call);                                            \
        if (err != cudaSuccess) {                                            \
            printf("Errore CUDA %s:%d: %s\n", __FILE__, __LINE__,            \
                   cudaGetErrorString(err));                                 \
            exit(1);                                                         \
        }                                                                    \
    } while (0)

#define BLOCK 8  // blocco 8 x 8 x 8 = 512 nodi, un thread per nodo

// Risolutore locale in shared memory: i 6 vicini del nodo sono nella copia s (blocco + halo)
__device__ float solve_in_block(const float s[BLOCK + 2][BLOCK + 2][BLOCK + 2], int z, int y, int x, float F, float h)
{
    float a = fminf(s[z][y][x - 1], s[z][y][x + 1]);               // a <- min dei vicini lungo x
    float b = fminf(s[z][y - 1][x], s[z][y + 1][x]);               // b <- min dei vicini lungo y
    float c = fminf(s[z - 1][y][x], s[z + 1][y][x]);               // c <- min dei vicini lungo z
    return fminf(s[z][y][x], solve_quadratic3(a, b, c, F, h));      // return min(u_ijk, u)
}

// Elabora un blocco di 8^3 nodi (512 thread, uno per nodo), cuore dell'Alg. 7:
//  1) copia blocco + halo in shared memory;
//  2) niter iterazioni locali su tutti i nodi insieme;
//  3) riscrive in u i propri nodi;
//  4) reduction: convergente se nessun nodo e' cambiato.
// Step 1: niter iterazioni; Step 2: 1 iterazione di prova.
__device__ void process_block(int block, int niter, float* u, const float* F,
                             int N, int nbx, float h, int src, float eps, unsigned char* Cb_out)
{
    __shared__ float s_u[BLOCK + 2][BLOCK + 2][BLOCK + 2];                // blocco + halo (10^3 valori), nella shared memory
    __shared__ unsigned char s_c[BLOCK * BLOCK * BLOCK];                  // i flag Cn dei 512 nodi

    int bi = (block / nbx) % nbx, bj = block % nbx, bk = block / (nbx * nbx);            // riga, colonna e strato del blocco
    int tx = threadIdx.x, ty = threadIdx.y, tz = threadIdx.z;
    int tid = (tz * BLOCK + ty) * BLOCK + tx;                                       // numero del thread nel blocco: 0..511
    int i = bi * BLOCK + ty, j = bj * BLOCK + tx, k = bk * BLOCK + tz;              // nodo di questo thread
    int idx = (k * N + i) * N + j;                                                  // posizione del nodo nella griglia
    bool inside = (i < N && j < N && k < N);                          // l'ultimo blocco puo' sporgere
    bool fixed  = !inside || (idx == src);             // nodo da non aggiornare: fuori dalla griglia o sorgente (u = 0)
    float f = inside ? F[idx] : 1.0f;                  // F_ijk del nodo

    // 1) Global -> Shared: 1000 celle (blocco + halo), 512 thread
    for (int m = tid; m < (BLOCK + 2) * (BLOCK + 2) * (BLOCK + 2); m += BLOCK * BLOCK * BLOCK) {
        int sz = m / ((BLOCK + 2) * (BLOCK + 2));
        int sy = (m / (BLOCK + 2)) % (BLOCK + 2);
        int sx = m % (BLOCK + 2);                             // cella (sz, sy, sx) della shared memory
        int gi = bi * BLOCK + sy - 1;                         // nodo della griglia corrispondente:
        int gj = bj * BLOCK + sx - 1;                         
        int gk = bk * BLOCK + sz - 1;                         // -1 perche' lo strato/riga/colonna 0 e' l'halo
        bool in_domain = (gi >= 0 && gi < N && gj >= 0 && gj < N && gk >= 0 && gk < N);
        s_u[sz][sy][sx] = in_domain ? u[(gk * N + gi) * N + gj] : INF;      // fuori dal dominio: INF
    }
    __syncthreads();                                         // la copia e' completa prima di usarla

    // 2) niter iterazioni locali in shared memory
    bool conv = true;                                        // Cn del nodo
    for (int it = 0; it < niter; ++it) {
        float p = s_u[tz+1][ty + 1][tx + 1];                 // valore prima dell'aggiornamento
        float q = fixed ? p                                  // sorgente / fuori: non cambia
                        : solve_in_block(s_u, tz+1, ty + 1, tx + 1, f, h);
        __syncthreads();                                     // (A) tutti hanno letto i vicini...
        s_u[tz+1][ty + 1][tx + 1] = q;
        __syncthreads();                                     // (B) ...e tutti hanno scritto
        conv = fabsf(p - q) < eps;                           // Cn: il nodo non e' cambiato
    }

    // 3) Shared -> Global: ogni thread riscrive il proprio nodo (non l'halo)
    if (inside) u[idx] = s_u[tz+1][ty + 1][tx + 1];

    // 4) Cb(b) <- reduction(Cn(b)): AND dei 512 flag
    s_c[tid] = conv;
    __syncthreads();
    for (int stride = BLOCK * BLOCK * BLOCK / 2; stride > 0; stride /= 2) {     // 256, ..., 4, 2, 1
        if (tid < stride) s_c[tid] = s_c[tid] && s_c[tid + stride];
        __syncthreads();
    }
    if (tid == 0) Cb_out[block] = s_c[0];                     // vero solo se TUTTI i nodi sono convergenti
}

// Step 1 (Alg. 7): ogni blocco della lista A fa niter iterazioni
__global__ void update_active_blocks(const int* list, float* u, const float* F,
                                    int N, int nbx, float h, int src, int niter,
                                    float eps, unsigned char* Cb1)
{
    process_block(list[blockIdx.x], niter, u, F, N, nbx, h, src, eps, Cb1);
}

// Inizio di ogni giro: costruisce la lista A dei blocchi attivi (Cb = 0)
__global__ void build_active_list(const unsigned char* Cb, unsigned char* active,
                                  int* list, int* count, int n_blocks) 
{
    int t = blockIdx.x * blockDim.x + threadIdx.x;           // numero del blocco di nodi
    if (t >= n_blocks) return;                               // thread in eccesso
    bool act = !Cb[t];                                       // attivo se NON convergente
    active[t] = act;
    if (act) list[atomicAdd(count, 1)] = t;                  // aggiungi t alla lista A
}

// Step 2, parte 1: i 6 vicini dei blocchi appena convergenti diventano candidati
__global__ void build_candidate_list(const int* list, int count, int nbx, const unsigned char* active,
                                     const unsigned char* Cb1, unsigned char* Cb,
                                     int* cand, int* cand_list, int* cand_count)
{
    int t = blockIdx.x * blockDim.x + threadIdx.x;           // blockDim = 256
    if (t >= count) return;                                  // thread in eccesso
    int b = list[t];                                         // un blocco della lista A
    Cb[b] = Cb1[b];                                          // non convergente (Caso 1): resta in A
    if (!Cb1[b]) return;                                     // proseguono solo i blocchi appena convergenti

    int bk = b / (nbx * nbx), bi = (b / nbx) % nbx, bj = b % nbx;                          // strato, riga e colonna del blocco
    int  nb[6] = { b - 1,  b + 1,       b - nbx, b + nbx,  b - nbx * nbx, b + nbx * nbx};  // sinistra, destra, sotto, sopra e davanti, dietro
    bool ok[6] = { bj > 0, bj < nbx - 1, bi > 0,  bi < nbx - 1, bk > 0, bk < nbx - 1 };
    for (int d = 0; d < 6; ++d) {
        if (!ok[d]) continue;                                // fuori dalla griglia di blocchi
        if (active[nb[d]] && !Cb1[nb[d]]) continue;          // Caso 1: lo aggiorna già lo Step 1
        if (atomicExch(&cand[nb[d]], 1) == 0)                // primo a segnarlo: niente doppioni
            cand_list[atomicAdd(cand_count, 1)] = nb[d];
    }
}

// Step 2, parte 2: ogni candidato fa 1 iterazione; se cambia, diventa attivo
__global__ void update_candidate_blocks(const int* cand_list, const int* cand_count, int* cand,
                                        float* u, const float* F, int N, int nbx, float h,
                                        int src, float eps, unsigned char* Cb)
{
    if (blockIdx.x >= *cand_count) return;                   // blocco CUDA in eccesso
    int b = cand_list[blockIdx.x];
    process_block(b, 1, u, F, N, nbx, h, src, eps, Cb);      // test del blocco: 1 iterazione
    if (threadIdx.x == 0 && threadIdx.y == 0 && threadIdx.z == 0) cand[b] = 0;   // pronto per il giro successivo
}

// Block FIM (Alg. 7) sulla GPU: restituisce u, il tempo del ciclo (ms) e il numero di giri
// Ogni giro: lista A -> Step 1 -> Step 2; fine quando A e' vuota
int block_fim(std::vector<float>& u, const std::vector<float>& F, int N, float h,
              int src, float eps, int niter, float* ms,
              std::vector<std::vector<int>>* attivi = nullptr, std::vector<std::vector<int>>* candidati = nullptr)
{
    const int nbx = (N + BLOCK - 1) / BLOCK, n_blocks = nbx * nbx * nbx; // blocchi per lato e blocchi totali
    const size_t bytes = (size_t)N * N * N * sizeof(float); // dimensione in byte della griglia

    std::vector<unsigned char> Cb0(n_blocks, 1);
    const int sk = src / (N * N), si = (src / N) % N, sj = src % N;   // strato, riga e colonna del nodo sorgente
    Cb0[((sk / BLOCK) * nbx + si / BLOCK) * nbx + sj / BLOCK] = 0;   // vale 1 ovunque tranne che nel blocco della sorgente

    // Memoria sulla GPU e copia dei dati iniziali (CPU -> GPU)
    float *d_u, *d_F; unsigned char *d_Cb, *d_Cb1, *d_active; int *d_list, *d_count;
    int *d_cand, *d_cand_list, *d_cand_count;
    CUDA_CHECK(cudaMalloc(&d_u, bytes));
    CUDA_CHECK(cudaMalloc(&d_F, bytes));
    CUDA_CHECK(cudaMalloc(&d_Cb, n_blocks));
    CUDA_CHECK(cudaMalloc(&d_Cb1, n_blocks));
    CUDA_CHECK(cudaMalloc(&d_active, n_blocks));
    CUDA_CHECK(cudaMalloc(&d_list, n_blocks * sizeof(int)));
    CUDA_CHECK(cudaMalloc(&d_count, sizeof(int)));

    // Memoria per i blocchi candidati
    CUDA_CHECK(cudaMalloc(&d_cand, n_blocks * sizeof(int)));
    CUDA_CHECK(cudaMalloc(&d_cand_list, n_blocks * sizeof(int)));
    CUDA_CHECK(cudaMalloc(&d_cand_count, sizeof(int)));

    CUDA_CHECK(cudaMemcpy(d_u, u.data(), bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_F, F.data(), bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_Cb, Cb0.data(), n_blocks, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemset(d_Cb1, 0, n_blocks));
    CUDA_CHECK(cudaMemset(d_cand, 0, n_blocks * sizeof(int)));


    const int threads = 256, list_blocks = (n_blocks + threads - 1) / threads;   // kernel "di servizio": 256 thread, un thread per blocco di nodi
    cudaEvent_t start, stop;
    CUDA_CHECK(cudaEventCreate(&start));
    CUDA_CHECK(cudaEventCreate(&stop));
    CUDA_CHECK(cudaEventRecord(start));                      // "segna l'ora" prima del ciclo (si misura solo il ciclo)
    int iter = 0;
    while (true) {                                           // un giro dell'Alg. 7
        int count = 0;
        CUDA_CHECK(cudaMemset(d_count, 0, sizeof(int)));
        build_active_list<<<list_blocks, threads>>>(d_Cb, d_active, d_list, d_count, n_blocks);
        CUDA_CHECK(cudaGetLastError());
        CUDA_CHECK(cudaMemcpy(&count, d_count, sizeof(int), cudaMemcpyDeviceToHost));
        if (count == 0) break;  // se count == 0, lista A vuota, termina ciclo
        ++iter;
        if (attivi) {                                        // solo per la visualizzazione: i blocchi di A in questo giro
            std::vector<int> A(count);
            CUDA_CHECK(cudaMemcpy(A.data(), d_list, count * sizeof(int), cudaMemcpyDeviceToHost));
            attivi->push_back(A);
        }
        // Step 1: foreach b in A in parallelo
        update_active_blocks<<<count, dim3(BLOCK, BLOCK, BLOCK)>>>(d_list, d_u, d_F, N, nbx, h, src, niter, eps, d_Cb1);    // si lanciano count blocchi CUDA, ognuno con 512 thread
        CUDA_CHECK(cudaGetLastError());
        // Step 2: solo i vicini dei blocchi appena convergenti (i candidati)
        CUDA_CHECK(cudaMemset(d_cand_count, 0, sizeof(int)));
        build_candidate_list<<<(count + threads - 1) / threads, threads>>>(d_list, count, nbx, d_active, d_Cb1, d_Cb,
                                                                          d_cand, d_cand_list, d_cand_count);
        CUDA_CHECK(cudaGetLastError());
        update_candidate_blocks<<<std::min(6 * count, n_blocks), dim3(BLOCK, BLOCK, BLOCK)>>>(d_cand_list, d_cand_count, d_cand,
                                                                                    d_u, d_F, N, nbx, h, src, eps, d_Cb);
        CUDA_CHECK(cudaGetLastError());
        if (candidati) {                                     // solo per la visualizzazione: i candidati dello Step 2
            int nc = 0;
            CUDA_CHECK(cudaMemcpy(&nc, d_cand_count, sizeof(int), cudaMemcpyDeviceToHost));
            std::vector<int> C(nc);
            CUDA_CHECK(cudaMemcpy(C.data(), d_cand_list, nc * sizeof(int), cudaMemcpyDeviceToHost));
            candidati->push_back(C);
        }
    }
    CUDA_CHECK(cudaEventRecord(stop));                       // "segna l'ora" dopo il ciclo
    CUDA_CHECK(cudaEventSynchronize(stop));                  // aspetta che la GPU ci arrivi
    CUDA_CHECK(cudaEventElapsedTime(ms, start, stop));       // tempo trascorso [ms]
    CUDA_CHECK(cudaEventDestroy(start));
    CUDA_CHECK(cudaEventDestroy(stop));

    // Risultato sulla CPU (GPU -> CPU) e liberazione della memoria
    CUDA_CHECK(cudaMemcpy(u.data(), d_u, bytes, cudaMemcpyDeviceToHost));
    CUDA_CHECK(cudaFree(d_u)); CUDA_CHECK(cudaFree(d_F));CUDA_CHECK(cudaFree(d_Cb));
    CUDA_CHECK(cudaFree(d_Cb1));
    CUDA_CHECK(cudaFree(d_active)); CUDA_CHECK(cudaFree(d_list)); CUDA_CHECK(cudaFree(d_count));
    CUDA_CHECK(cudaFree(d_cand)); CUDA_CHECK(cudaFree(d_cand_list)); CUDA_CHECK(cudaFree(d_cand_count));
    return iter;
}

In [ ]:
%%writefile main.cu
// Block FIM (Alg. 7) in 3D sulla GPU, da solo, nel caso test della sfera: F = 1, nessun ostacolo, sorgente al centro.
// La soluzione esatta e' la distanza dal centro, u = sqrt(x^2 + y^2 + z^2): per ogni N si misurano tempo e accuratezza.
#include <cstdio>
#include <algorithm>       // std::min
#include <fstream>         // file per la vista 3D (Python)
#include "eikonale.h"      // risolutore locale ed errore relativo
#include "block_fim.cuh"   // Block FIM (Alg. 7)

int main()
{
    cudaDeviceProp prop;
    CUDA_CHECK(cudaGetDeviceProperties(&prop, 0));
    printf("GPU: %s | multiprocessori: %d\n", prop.name, prop.multiProcessorCount);

    CUDA_CHECK(cudaFree(0));                                 // inizializza la GPU prima delle misure
    const float eps = 1e-6f;                                 // tolleranza di convergenza
    const int niter = 12;                                    // iterazioni locali per visita di un blocco
    const int ripetizioni = 3;                               // ogni tempo = il minimo di 3 misure
    printf("\n    N   nodi [milioni]   passo h   giri   t Block FIM [ms]   errore relativo   errore massimo\n");

    for (int Nk : { 33, 65, 129, 193, 257 }) {
        const float hk = 2.0f / (Nk - 1);                    // passo della griglia nel cubo [-1, 1]^3
        const int n = Nk * Nk * Nk;
        const int srck = ((Nk / 2) * Nk + Nk / 2) * Nk + (Nk / 2);   // sorgente al centro
        const std::vector<float> Fk(n, 1.0f);                // F = 1: u = distanza dal centro
        std::vector<float> u0(n, INF);                       // condizione iniziale: 0 nella sorgente, INF altrove
        u0[srck] = 0.0f;

        std::vector<float> u;
        double t = 1e30;
        int giri = 0;
        for (int r = 0; r < ripetizioni; ++r) {              // 3 misure, si tiene la piu' veloce
            float ms = 0.0f;
            u = u0;
            giri = block_fim(u, Fk, Nk, hk, srck, eps, niter, &ms);    // GPU: Alg. 7
            t = std::min(t, (double)ms);
        }

        double err_max = 0.0;                                // errore massimo |u - esatta| su tutta la griglia
        for (int k = 0; k < Nk; ++k)
            for (int i = 0; i < Nk; ++i)
                for (int j = 0; j < Nk; ++j) {
                    const double x = -1.0 + j * hk, y = -1.0 + i * hk, z = -1.0 + k * hk;
                    err_max = std::max(err_max, std::fabs(u[(k * Nk + i) * Nk + j] - std::sqrt(x * x + y * y + z * z)));
                }
        printf("%5d   %14.2f   %7.4f   %4d   %16.2f   %15.3e   %14.3e\n", Nk, n / 1e6, hk, giri, t,
               relative_error(u, Nk, hk), err_max);

        if (Nk == 65) {                                      // soluzione per la vista 3D (Python)
            std::ofstream out("fim_65.bin", std::ios::binary);
            out.write(reinterpret_cast<const char*>(u.data()), u.size() * sizeof(float));
        }
    }
    // Una esecuzione in piu' (non cronometrata) a N = 65 che registra, giro per giro, i blocchi attivi e i candidati
    {
        const int Nk = 65, n = Nk * Nk * Nk, srck = ((Nk / 2) * Nk + Nk / 2) * Nk + (Nk / 2);
        std::vector<float> u(n, INF), Fk(n, 1.0f);
        u[srck] = 0.0f;
        std::vector<std::vector<int>> attivi, candidati;
        float ms = 0.0f;
        block_fim(u, Fk, Nk, 2.0f / (Nk - 1), srck, eps, niter, &ms, &attivi, &candidati);
        std::ofstream out("blocchi_65.txt");                 // due righe per giro: "A b1 b2 ..." e "C c1 c2 ..."
        for (size_t g = 0; g < attivi.size(); ++g) {
            out << "A";
            for (int b : attivi[g]) out << " " << b;
            out << "\nC";
            for (int b : candidati[g]) out << " " << b;
            out << "\n";
        }
        printf("\nblocchi attivi per giro (N = 65) salvati in blocchi_65.txt: %zu giri\n", attivi.size());
    }
    return 0;
}

In [ ]:
!nvcc -O3 -arch=sm_75 main.cu -o block_fim && ./block_fim

In [ ]:
# Vista 3D: il fronte u = t del Block FIM (colori) si espande sulla sfera esatta di raggio t (reticolo), N = 65.
# I colori sono lo scarto tra il raggio del Block FIM e quello esatto; lo slider (o Play) fa avanzare t.
import numpy as np
import plotly.graph_objects as go
from skimage.measure import marching_cubes                          # superficie u = t dal cubo di valori

N = 65
h = 2.0 / (N - 1)
u = np.fromfile("fim_65.bin", dtype=np.float32).reshape(N, N, N)   # u[k, i, j]: strato k = z, riga i = y, colonna j = x
istanti = np.round(np.linspace(0.1, 0.9, 9), 2)                     # istanti t (fronti dentro il cubo [-1, 1]^3)

def fronte(t):                                                      # superficie u = t del FIM, colorata con lo scarto dalla sfera
    v, f, _, _ = marching_cubes(u, level=t, spacing=(h, h, h))
    p = v - 1.0                                                     # vertici in [-1, 1], ordine (z, y, x)
    dr = np.linalg.norm(p, axis=1) - t                              # raggio del FIM - raggio esatto, per ogni vertice
    mesh = go.Mesh3d(x=p[:, 2], y=p[:, 1], z=p[:, 0], i=f[:, 0], j=f[:, 1], k=f[:, 2],
                     intensity=dr, cmin=-0.04, cmax=0.0, colorscale="Viridis", opacity=0.9,
                     colorbar=dict(title="raggio FIM -<br>raggio esatto"), name="fronte del FIM")
    return mesh, dr

def sfera(t):                                                       # reticolo della sfera esatta di raggio t
    th, ph = np.meshgrid(np.linspace(0, np.pi, 13), np.linspace(0, 2 * np.pi, 25), indexing="ij")
    X, Y, Z = t * np.sin(th) * np.cos(ph), t * np.sin(th) * np.sin(ph), t * np.cos(th)
    xs, ys, zs = [], [], []
    for a, b, c in [(X, Y, Z), (X.T, Y.T, Z.T)]:                  # paralleli e meridiani, separati da None
        for r in range(a.shape[0]):
            xs += list(a[r]) + [None]; ys += list(b[r]) + [None]; zs += list(c[r]) + [None]
    return go.Scatter3d(x=xs, y=ys, z=zs, mode="lines", line=dict(color="black", width=1), name="sfera esatta")

for t in istanti:                                                   # verifica numerica: quanto il fronte si scosta dalla sfera
    _, dr = fronte(t)
    print(f"t = {t:.2f}: raggio del FIM da {t + dr.min():.4f} a {t + dr.max():.4f} (esatto {t:.2f})")

fig = go.Figure(data=[fronte(istanti[0])[0], sfera(istanti[0])],
                frames=[go.Frame(data=[fronte(t)[0], sfera(t)], traces=[0, 1], name=f"{t:.2f}") for t in istanti])
fig.update_layout(
    title="Block FIM 3D, F = 1: il fronte u = t (colori) si espande sulla sfera esatta (reticolo)",
    scene=dict(xaxis=dict(range=[-1, 1]), yaxis=dict(range=[-1, 1]), zaxis=dict(range=[-1, 1]), aspectmode="cube"),
    height=750, margin=dict(l=0, r=0, t=40, b=80),
    updatemenus=[dict(type="buttons", direction="left", x=0.1, y=0, xanchor="right", yanchor="top", pad=dict(r=10, t=70), buttons=[
        dict(label="Play", method="animate",
             args=[None, dict(frame=dict(duration=500, redraw=True), fromcurrent=True, transition=dict(duration=0))]),
        dict(label="Pausa", method="animate",
             args=[[None], dict(frame=dict(duration=0, redraw=False), mode="immediate")])])],
    sliders=[dict(x=0.1, len=0.9, y=0, yanchor="top", currentvalue=dict(prefix="fronte u = t = "), pad=dict(t=50),
                  steps=[dict(label=f"{t:.2f}", method="animate",
                              args=[[f"{t:.2f}"], dict(mode="immediate", frame=dict(duration=0, redraw=True))])
                         for t in istanti])])
fig.write_html("fronte_sfera.html", include_plotlyjs="cdn")         # anche come file da scaricare: si apre nel browser
fig.show()

In [ ]:
# Il Block FIM al lavoro (N = 65, sorgente al centro): ogni frame e' un giro dell'Alg. 7.
# Blu = blocchi attivi (lista A, Step 1); grigio = blocchi gia' visitati; arancione = candidati dello Step 2
# (nascosti all'inizio: si accendono cliccando sulla legenda).
# Si mostra solo meta' cubo (y <= 0), come un taglio, per vedere dentro il guscio di blocchi.
# Sotto: quanti blocchi attivi e candidati ci sono in ogni giro (in tutto il cubo).
import numpy as np
import plotly.graph_objects as go

N, BLOCK = 65, 8
h = 2.0 / (N - 1)
nbx = (N + BLOCK - 1) // BLOCK                                      # blocchi per lato (9 x 9 x 9 = 729)
righe = [r.split() for r in open("blocchi_65.txt")]
attivi = [[int(b) for b in r[1:]] for r in righe[0::2]]             # riga "A ..." di ogni giro
candidati = [[int(b) for b in r[1:]] for r in righe[1::2]]          # riga "C ..." di ogni giro
BLU, ARANCIO, GRIGIO = "#2a78d6", "#eb6834", "#b4b2a9"

def cubi(blocchi, colore, opacita, nome):                           # un unico Mesh3d con un cubetto per ogni blocco
    X, Y, Z, I, J, K = [], [], [], [], [], []
    for b in blocchi:
        bk, bi, bj = b // (nbx * nbx), (b // nbx) % nbx, b % nbx    # strato, riga, colonna del blocco
        if bi > nbx // 2: continue                                  # taglio: si tengono solo i blocchi con y <= 0
        lo = -1.0 + h * np.array([bj, bi, bk]) * BLOCK              # angolo del blocco (x, y, z)
        hi = np.minimum(lo + h * (BLOCK - 1), 1.0)                  # l'ultimo blocco sporge: si taglia a 1
        g = 0.12 * h * BLOCK                                        # piccolo spazio tra cubetti vicini
        x0, y0, z0 = lo + g / 2; x1, y1, z1 = hi - g / 2 + h
        x1, y1, z1 = min(x1, 1.0), min(y1, 1.0), min(z1, 1.0)
        o = len(X)
        X += [x0, x1, x1, x0, x0, x1, x1, x0]; Y += [y0, y0, y1, y1, y0, y0, y1, y1]; Z += [z0, z0, z0, z0, z1, z1, z1, z1]
        for a, b2, c in [(0, 1, 2), (0, 2, 3), (4, 5, 6), (4, 6, 7), (0, 1, 5), (0, 5, 4),
                         (1, 2, 6), (1, 6, 5), (2, 3, 7), (2, 7, 6), (3, 0, 4), (3, 4, 7)]:
            I.append(o + a); J.append(o + b2); K.append(o + c)
    return go.Mesh3d(x=X, y=Y, z=Z, i=I, j=J, k=K, color=colore, opacity=opacita, flatshading=True, name=nome)

def frame(g):                                                       # le tre famiglie di blocchi al giro g
    visti = set(b for gg in range(g) for b in attivi[gg]) - set(attivi[g]) - set(candidati[g])
    return [cubi(sorted(visti), GRIGIO, 0.08, "gia' visitati"), cubi(attivi[g], BLU, 0.9, "attivi (Step 1)"),
            cubi(candidati[g], ARANCIO, 0.25, "candidati (Step 2)")]

fig = go.Figure(data=frame(0), frames=[go.Frame(data=frame(g), traces=[0, 1, 2], name=str(g + 1)) for g in range(len(attivi))])
fig.data[2].visible = "legendonly"                                  # candidati: si mostrano cliccando sulla legenda
fig.update_layout(
    title="Block FIM 3D: i blocchi 8x8x8 giro per giro (N = 65, meta' cubo tagliata a y = 0)",
    scene=dict(xaxis=dict(range=[-1, 1]), yaxis=dict(range=[-1, 1]), zaxis=dict(range=[-1, 1]), aspectmode="cube",
               camera=dict(eye=dict(x=1.0, y=1.9, z=0.8))),               # si guarda il taglio y = 0 da davanti
    height=720, margin=dict(l=0, r=0, t=40, b=80), legend=dict(x=0.01, y=0.95),
    updatemenus=[dict(type="buttons", direction="left", x=0.1, y=0, xanchor="right", yanchor="top", pad=dict(r=10, t=70), buttons=[
        dict(label="Play", method="animate",
             args=[None, dict(frame=dict(duration=600, redraw=True), fromcurrent=True, transition=dict(duration=0))]),
        dict(label="Pausa", method="animate",
             args=[[None], dict(frame=dict(duration=0, redraw=False), mode="immediate")])])],
    sliders=[dict(x=0.1, len=0.9, y=0, yanchor="top", currentvalue=dict(prefix="giro "), pad=dict(t=50),
                  steps=[dict(label=str(g + 1), method="animate",
                              args=[[str(g + 1)], dict(mode="immediate", frame=dict(duration=0, redraw=True))])
                         for g in range(len(attivi))])])
fig.write_html("blocchi_attivi.html", include_plotlyjs="cdn")
fig.show()

giri = np.arange(1, len(attivi) + 1)
conta = go.Figure([go.Scatter(x=giri, y=[len(a) for a in attivi], mode="lines+markers", name="blocchi attivi (Step 1)",
                              line=dict(color=BLU, width=2), marker=dict(size=8)),
                   go.Scatter(x=giri, y=[len(c) for c in candidati], mode="lines+markers", name="candidati (Step 2)",
                              line=dict(color=ARANCIO, width=2), marker=dict(size=8))])
conta.update_layout(template="plotly_white", height=380, title="Quanti blocchi lavorano in ogni giro (N = 65)",
                    xaxis_title="giro", yaxis_title="numero di blocchi", legend=dict(orientation="h", y=1.12))
conta.show()